In [ ]:
import matplotlib as mpl

mpl.rcParams.update({

    # -------- Lines --------
    "lines.linewidth": 2,
    "lines.markersize": 4.5,

    # -------- Axes labels --------
    "axes.labelsize": 14,
    "axes.titlesize": 14,

    # -------- Tick labels --------
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    # -------- Legend --------
    "legend.fontsize": 11,

    # -------- Grid --------
    "grid.alpha": 0.25,

})

plot A function

In [ ]:
def plot_A(ax):
    """
    Biomorph — Plot A (self-contained)
    Fixed bin width = 15 with adaptive log inset floor.
    """

    import numpy as np
    import pandas as pd
    from pathlib import Path
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    # ===================================
    # Load data
    # ===================================

    base_path = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/biomorph/"
        "plot_a/plot_a_files"
    )

    g_full_file = base_path / "unified_plotA.txt"
    p_file = base_path / "P_dist.txt"

    df_g = pd.read_csv(g_full_file, sep=r"\s+")
    df_p = pd.read_csv(p_file, sep=r"\s+")

    g_raw = pd.to_numeric(df_g["complexity"], errors="coerce").dropna().to_numpy()
    g_raw = g_raw[g_raw > 0]

    p_raw = pd.to_numeric(df_p["complexity"], errors="coerce").dropna().to_numpy()
    p_raw = p_raw[p_raw > 0]

    g_complexity_mean = np.mean(g_raw)
    p_complexity_mean = np.mean(p_raw)
    print(f"G-dist mean complexity: {g_complexity_mean:.6f}")
    print(f"P-dist mean complexity: {p_complexity_mean:.6f}")
    #print number of samples in G-dist and P-dist
    print(f"G-dist sample size: {len(g_raw)}")
    print(f"P-dist sample size: {len(p_raw)}")

    # ===================================
    # Shared range
    # ===================================

    all_data = np.concatenate([g_raw, p_raw])
    min_val = float(np.min(all_data))
    max_val = float(np.max(all_data))

    # ===================================
    # Fixed bin width
    # ===================================

    bin_width = 15
    bins = np.arange(min_val, max_val + bin_width, bin_width)
    n_bins = len(bins) - 1

    g_counts, bin_edges = np.histogram(g_raw, bins=bins)
    p_counts, _ = np.histogram(p_raw, bins=bins)

    g_counts = g_counts / g_counts.sum()
    p_counts = p_counts / p_counts.sum()

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    # ===================================
    # Adaptive epsilon floor
    # ===================================

    all_nonzero = np.concatenate([
        g_counts[g_counts > 0],
        p_counts[p_counts > 0]
    ])

    eps = float(np.min(all_nonzero) * 0.1) if all_nonzero.size else 1e-12

    g_safe = np.clip(g_counts, eps, None)
    p_safe = np.clip(p_counts, eps, None)

    # ===================================
    # Main Plot
    # ===================================

    ax.plot(bin_centers, g_counts,
            marker="o",
            color="#FF8C00",
            label="G-dist")

    ax.fill_between(bin_centers, g_counts, 0,
                    color="#FF8C00",
                    alpha=0.18)

    ax.plot(bin_centers, p_counts,
            marker="o",
            color="blue",
            label="P-dist")

    ax.fill_between(bin_centers, p_counts, 0,
                    color="blue",
                    alpha=0.15)

    ax.set_xlabel("Complexity (Lempel-Ziv)")
    ax.set_ylabel("Relative Frequency")

    ax.legend(frameon=False, loc="upper left")

    #ax.grid(True, which="major", linestyle="-", alpha=0.2)
    ax.set_axisbelow(True)
    ax.minorticks_off()

    # ===================================
    # Inset
    # ===================================

    ax_inset = inset_axes(ax, width="22%", height="22%", loc="upper right")

    ax_inset.plot(bin_centers, g_safe, color="#FF8C00")
    ax_inset.plot(bin_centers, p_safe, color="blue")

    ax_inset.set_yscale("log")
    ax_inset.set_xlabel("Complexity", fontsize=9)
    ax_inset.set_ylabel("Log Frequency", fontsize=9)
    ax_inset.tick_params(labelsize=8)
    

    return dict(bin_width=bin_width, n_bins=n_bins, epsilon=eps)

plot B function

In [ ]:
def plot_B(ax):
    """
    Biomorph — Plot B
    Entropy vs scaled mean complexity (global scaling).
    """

    import pandas as pd
    from pathlib import Path

    # --------------------------------------------------
    # Load processed biomorph data
    # --------------------------------------------------

    data_file = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/biomorph/"
        "plot_b/plot_b_files/plotB_biomorph_entropy_global_vs_local.txt"
    )

    df = pd.read_csv(data_file, sep=r"\s+")

    N_values = df["sample_size"].values
    entropy = df["entropy"].values
    scaled_global = df["scaled_global"].values

    # --------------------------------------------------
    # Plot
    # --------------------------------------------------

    # Entropy (green)
    ax.semilogx(
        N_values, entropy,
        'o-',
        color='green',
        label=r'$S$'
    )

    # Global scaling (orange)
    ax.semilogx(
        N_values, scaled_global,
        'o-',
        color='orange',
        label=r'$\langle \tilde{K}_{g} \rangle$'
    )

    ax.set_xlabel('Number of sampled genotypes (N, log scale)')
    ax.set_ylabel(r'Entropy (S) and scaled $\langle \tilde{K}_{g} \rangle$')
    ax.set_xscale("log")
    ax.minorticks_off()

    ax.grid(True, which="major", linestyle="-", alpha=0.2)

    ax.legend(frameon=False)

plot C function

In [ ]:
def plot_C(ax):
    """
    Biomorph — Plot C
    Mean KC vs Shannon entropy (linear regression).
    """

    import numpy as np
    import pandas as pd
    from pathlib import Path

    data_path = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/biomorph/"
        "plot_c/plotC_data.txt"
    )

    df = pd.read_csv(data_path, sep=r"\s+")

    df["genome_size"] = pd.to_numeric(df["genome_size"], errors="coerce")
    df["KC"] = pd.to_numeric(df["KC"], errors="coerce")
    df["shannon_entropy"] = pd.to_numeric(df["shannon_entropy"], errors="coerce")

    df = df.dropna()

    entropy_vals = df["shannon_entropy"].values
    kc_vals = df["KC"].values
    labels = df["genome_size"].astype(int).astype(str).values

    # Linear regression
    m, b = np.polyfit(entropy_vals, kc_vals, 1)
    fit_line = m * entropy_vals + b
    sign = "-" if b < 0 else "+"
    eq_label = rf"$\langle \tilde{{K}}_{{g}} \rangle = {m:.2f}\,S {sign} {abs(b):.2f}$"

    ax.scatter(entropy_vals, kc_vals, color='orange')
    ax.plot(entropy_vals, fit_line, color='orange', label=eq_label)

    for xi, yi, lbl in zip(entropy_vals, kc_vals, labels):
        ax.text(xi, yi, lbl, fontsize=11, ha='center', va='bottom')

    ax.set_xlabel("Shannon Entropy (S)")
    ax.set_ylabel(r"Mean Complexity $\langle \tilde{K}_{g} \rangle$")
    #ax.grid(True, which="major", linestyle="-", alpha=0.2)
    ax.margins(x=0.05, y=0.05)
    ax.legend(frameon=False)

plot D function

In [ ]:
def plot_D(ax):
    """
    Biomorph — Plot D
    Mean KC vs mutation number grouped by quintile.
    """

    import numpy as np
    import pandas as pd
    from pathlib import Path

    data_path = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/circadian/"
        "mut_project_updates/figures/biomorph/plot_d/plotD_data_incl0.txt"
    )

    marker = "o"

    # --------------------------------
    # Load data
    # --------------------------------
    df = pd.read_csv(data_path, sep=r"\s+")

    df["n_mutations"] = pd.to_numeric(df["n_mutations"], errors="coerce")
    df["mean_KC"] = pd.to_numeric(df["mean_KC"], errors="coerce")
    df["quintile"] = pd.to_numeric(df["quintile"], errors="coerce")

    df = df.dropna(subset=["n_mutations", "mean_KC", "quintile"])

    df["n_mutations"] = df["n_mutations"].astype(int)
    df["quintile"] = df["quintile"].astype(int)

    # Pivot table
    pivot = df.pivot_table(index="n_mutations",
                           columns="quintile",
                           values="mean_KC",
                           aggfunc="mean")

    pivot = pivot.sort_index()

    quints = sorted(pivot.columns)

    x_positions = np.arange(len(pivot.index))
    x_tick_labels = [str(int(x)) for x in pivot.index]

    # --------------------------------
    # Plot (pink → plum gradient G1 → G5)
    # --------------------------------
    palette = [
        "#E8B5D6",  # G1
        "#CC79A7",  # G2
        "#A63478",  # G3
        "#7A1F5C",  # G4
        "#4A0D3A",  # G5
    ]

    for i, q in enumerate(quints):
        ax.plot(x_positions,
                pivot[q].to_numpy(),
                marker=marker,
                color=palette[i],
                label=f"G{q}")

    ax.set_xlabel("Number of mutations")
    ax.set_ylabel(r"Mean Complexity $\langle \tilde{K}_{g} \rangle$")

    # show every other tick
    ax.set_xticks(x_positions[::2])
    ax.set_xticklabels(x_tick_labels[::2])

    ax.grid(True, which="major", linestyle="-", alpha=0.2)

    # Reverse legend (G5 → G1)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1],
              loc="upper right",
              frameon=False)

    # 0.5 tick spacing margin
    if len(x_positions) > 1:
        spacing = x_positions[1] - x_positions[0]
    else:
        spacing = 1.0

    ax.set_xlim(x_positions[0] - 0.5 * spacing,
                x_positions[-1] + 0.5 * spacing)

create master figure

In [ ]:
import matplotlib.pyplot as plt

# ===================================
# Create master 2x2 figure
# ===================================
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes = axes.flatten()

# Call your plotting functions
plot_A(axes[0])
plot_B(axes[1])
plot_C(axes[2])
plot_D(axes[3])

# ===================================
# Panel labels underneath (standardized)
# ===================================
panel_labels = ['(a)', '(b)', '(c)', '(d)']

for ax, label in zip(axes, panel_labels):
    ax.text(
        0.5,
        -0.14,
        label,
        transform=ax.transAxes,
        ha='center',
        va='top',
        fontsize=11
    )

plt.tight_layout()
plt.subplots_adjust(hspace=0.28, wspace=0.25)

plt.show()